In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
Path.cwd()

In [ ]:
neutron_data_path = Path.cwd().parents[2] / "Documents" / "Neutron Data"
neutron_data_path

In [ ]:
unconverted_data_path = neutron_data_path / "1-Unconverted_Data"
converted_data_path = neutron_data_path / "2-Converted_Data"

### CSV files (straight from Compass)

In [ ]:
exp_data_path = unconverted_data_path / "ID-289" / "UNFILTERED"

In [ ]:
csv_file_paths = [x for x in exp_data_path.iterdir() if x.suffix.lower() == '.csv']

In [ ]:
with open(csv_file_paths[0], 'r') as openfile:
    headers_line = openfile.readline()
    data_sample_line = openfile.readline()

col_names = headers_line.strip().split(';')[:-1]
cols_count = len(data_sample_line.strip().split(';'))
samples_count = cols_count - len(col_names)
samples_col_names = [str(i) for i in range(samples_count)]
col_names += samples_col_names

In [ ]:
dfs = []
total_rows = 0
for i, csv_path in enumerate(csv_file_paths):
    header = 0 if i == 0 else None
    df = pd.read_csv(csv_path, sep=';', header=0, names=col_names)
    rows = df.shape[0]
    print(rows)
    dfs.append(df)
    total_rows += rows

In [ ]:
total_rows

In [ ]:
full_df = pd.concat(dfs).sort_values("TIMETAG", ignore_index=True)
csv_rows = full_df.shape[0]
csv_rows

In [ ]:
full_df.dropna().shape[0]

### Old Converter Parquet Files

In [ ]:
# exp_data_path = converted_data_path/'testing'/'ID-289-original'/'processed_data'/'unfiltered'/'psd'
# parquet_file_paths = [x for x in exp_data_path.iterdir() if x.suffix.lower() == '.parquet']
# parquet_file_paths

In [ ]:
# dfs = []
# total_rows = 0
# for i, parquet_path in enumerate(parquet_file_paths):
#     header = 0 if i == 0 else None
#     df = pd.read_parquet(parquet_path)
#     rows = df.shape[0]
#     print(rows)
#     dfs.append(df)
#     total_rows += rows

In [ ]:
# total_rows

In [ ]:
# full_df_old = pd.concat(dfs)
# old_rows = full_df_old.shape[0]
# old_rows

In [ ]:
# full_df_old.dropna().shape[0]

In [ ]:
# full_df_old = pd.read_parquet(parquet_file_paths)
# full_df_old.shape[0]

In [ ]:
# full_df_old.dropna().shape[0]

### New Converter Parquet Files

In [ ]:
exp_data_path = converted_data_path/'ID-289'/'processed_data'/'unfiltered'/'psd'
parquet_file_paths = [x for x in exp_data_path.iterdir() if x.suffix.lower() == '.parquet']
parquet_file_paths

In [ ]:
dfs = []
total_rows = 0
for i, parquet_path in enumerate(parquet_file_paths):
    header = 0 if i == 0 else None
    df = pd.read_parquet(parquet_path)
    rows = df.shape[0]
    print(rows)
    dfs.append(df)
    total_rows += rows

In [ ]:
total_rows

In [ ]:
full_df_new = pd.concat(dfs).sort_values("TIMETAG", ignore_index=True)
new_rows = full_df_new.shape[0]
new_rows

In [ ]:
full_df_new.dropna().shape[0]

In [ ]:
full_df_new = pd.read_parquet(parquet_file_paths)
full_df_new.shape[0]

In [ ]:
full_df_new.dropna().shape[0]

### What's missing?

In [ ]:
csv_rows - new_rows

In [ ]:
csv_rows

In [ ]:
new_rows

In [ ]:
full_df.head()

In [ ]:
full_df_new.head()

In [ ]:
timetag_csv = full_df["TIMETAG"].sort_values(ignore_index=True)
print(timetag_csv.shape)
timetag_csv_prev = timetag_csv.shift().fillna(-1).astype("int64")
print(timetag_csv.head())
print(timetag_csv_prev.head())
timetag_ascending = timetag_csv <= timetag_csv_prev
timetag_ascending[timetag_ascending].index

In [ ]:
timetag_new = full_df_new["TIMETAG"].astype("int64").sort_values(ignore_index=True)
print(timetag_new.shape)
timetag_new_prev = timetag_new.shift().fillna(-1).astype("int64")
print(timetag_new.head())
print(timetag_new_prev.head())
timetag_ascending_new = timetag_new <= timetag_new_prev
print(timetag_ascending_new.shape)
bad_indices = timetag_ascending_new[timetag_ascending_new].index
print(bad_indices)
print(timetag_new[bad_indices])

In [ ]:
added_timetag = np.setdiff1d(timetag_new.to_numpy(), timetag_csv.to_numpy())
print(added_timetag)
print(list(added_timetag))
full_df_new[full_df_new["TIMETAG"].astype("int64").isin(added_timetag)]

In [ ]:
full_df.iloc[[215703, 215704, 215705]]

In [ ]:
full_df_new.iloc[[215703, 215704, 215705]]

In [ ]:
# https://stackoverflow.com/questions/61691228/element-wise-comparison-of-two-pandas-dataframes

for col_name in list(full_df_new.columns):
    csv_series = full_df[col_name]
    csv_dtype = csv_series.dtype
    new_series = full_df_new[col_name].astype(csv_dtype)
    csv_array = csv_series.to_numpy()
    new_array = new_series.to_numpy()
    print(f"Series: {col_name}")
    
    print(np.equal(csv_array, new_array))